# GMM with Risk-Adjusted Regime Signals

Assigns bullish and bearish regimes by Sharpe thresholds and validates trading variants.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
GMM Regime Detection Trading Framework - Multi-Instrument Adaptive Strategy
=============================================================================
Enhanced version that adapts instrument selection based on detected regime:
- BULLISH REGIME: Uses leveraged ETF (TQQQ) for amplified gains
- NEUTRAL REGIME: Uses standard ETF (SPY) without leverage
- BEARISH REGIME: Uses inverse ETF (SH) at 20% for hedging

Author: Trading Strategy Implementation
Date: 2025
Version: 2.0 - Multi-Instrument Adaptive
"""

from _market_data import download_price_records
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import warnings

warnings.filterwarnings("ignore")


class Config:
    """
    Centralized configuration for the entire trading framework.
    All parameters are exposed here to avoid hardcoding throughout the codebase.
    """

    START_DATE = "2019-01-01"
    END_DATE = "2024-12-31"
    FEATURE_TICKER = "SPY"
    BULLISH_INSTRUMENT = "TQQQ"
    NEUTRAL_INSTRUMENT = "SPY"
    BEARISH_INSTRUMENT = "SH"
    BULLISH_POSITION_SIZE = 1.0
    NEUTRAL_POSITION_SIZE = 1.0
    BEARISH_POSITION_SIZE = 0.2
    BULLISH_SHARPE_THRESHOLD = 0.5
    BEARISH_SHARPE_THRESHOLD = -0.2
    BENCHMARK_TICKER = "URTH"
    STOCHASTIC_PERIOD = 14
    STOCHASTIC_SMOOTH = 3
    GARMAN_KLASS_PERIOD = 20
    ZSCORE_LOOKBACK = 60
    N_COMPONENTS = 3
    GMM_COVARIANCE = "full"
    GMM_MAX_ITER = 100
    GMM_N_INIT = 10
    GMM_RANDOM_STATE = 42
    MIN_TRAINING_SAMPLES = 120
    RETRAIN_FREQUENCY = 20
    USE_REGIME_CONFIDENCE = True
    MIN_REGIME_PROBABILITY = 0.6
    COMMISSION_RATE = 0.001
    INITIAL_CAPITAL = 100000
    PLOT_RESULTS = True
    SAVE_RESULTS = True
    OUTPUT_DIR = "./gmm_strategy_results/"


class FMPDataFetcher:
    """
    Fetches OHLCV data from Yahoo Finance.
    Ensures no synthetic data or forward-filling is used.
    """

    def get_historical_data(
        self, ticker: str, start_date: str, end_date: str
    ) -> Optional[pd.DataFrame]:
        """
        Fetch historical OHLCV data for a single ticker.

        CRITICAL: No forward-filling or back-filling.
        Missing data remains NaN and strategy will not trade on those days.

        Parameters:
        -----------
        ticker : str
            Stock ticker symbol
        start_date : str
            Start date in YYYY-MM-DD format
        end_date : str
            End date in YYYY-MM-DD format

        Returns:
        --------
        pd.DataFrame : OHLCV data with DatetimeIndex, or None if fetch fails
        """
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) == 0:
                print(f"No data available for {ticker}")
                return None
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date")
            df.set_index("date", inplace=True)
            df = df[["open", "high", "low", "close", "volume"]].copy()
            df.columns = ["Open", "High", "Low", "Close", "Volume"]
            print(f"Fetched {len(df)} days of data for {ticker}")
            return df
        except Exception as e:
            print(f"Error fetching data for {ticker}: {e}")
            return None


class FeatureEngineer:
    """
    Calculates technical indicators without look-ahead bias.
    All calculations use only data available up to time t for decision at time t.

    CRITICAL: Features are calculated from FEATURE_TICKER (e.g., SPY)
    These features are used to detect regime, which then determines which
    instrument to trade (TQQQ, SPY, or SH).
    """

    @staticmethod
    def calculate_stochastic_k(df: pd.DataFrame, period: int = 14, smooth: int = 3) -> pd.Series:
        """
        Calculate Stochastic K oscillator.

        Formula:
        %K = 100 * (Close - Low_n) / (High_n - Low_n)
        where Low_n = lowest low over period, High_n = highest high over period

        Then apply SMA smoothing.

        CRITICAL: .rolling() naturally avoids look-ahead as it only uses past data

        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        period : int
            Lookback period for high/low calculation
        smooth : int
            Smoothing period (SMA)

        Returns:
        --------
        pd.Series : Smoothed Stochastic K values
        """
        low_min = df["Low"].rolling(window=period, min_periods=period).min()
        high_max = df["High"].rolling(window=period, min_periods=period).max()
        stoch_k = 100 * (df["Close"] - low_min) / (high_max - low_min)
        stoch_k_smooth = stoch_k.rolling(window=smooth, min_periods=smooth).mean()
        return stoch_k_smooth

    @staticmethod
    def calculate_garman_klass_volatility(df: pd.DataFrame, period: int = 20) -> pd.Series:
        """
        Calculate Garman-Klass volatility estimator.

        This is a more efficient volatility estimator than close-to-close,
        as it uses OHLC information.

        Formula:
        GK_vol = sqrt( (1/n) * sum( 0.5*(log(H/L))^2 - (2*log(2)-1)*(log(C/O))^2 ) )

        Then annualized: GK_vol * sqrt(252)

        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        period : int
            Rolling window for volatility calculation

        Returns:
        --------
        pd.Series : Annualized Garman-Klass volatility
        """
        hl = np.log(df["High"] / df["Low"])
        co = np.log(df["Close"] / df["Open"])
        gk_component = 0.5 * hl**2 - (2 * np.log(2) - 1) * co**2
        gk_vol = gk_component.rolling(window=period, min_periods=period).mean()
        gk_vol_annualized = np.sqrt(gk_vol * 252)
        return gk_vol_annualized

    @staticmethod
    def rolling_zscore(series: pd.Series, window: int = 60) -> pd.Series:
        """
        Calculate rolling z-score without look-ahead bias.

        Z-score = (X - rolling_mean) / rolling_std

        CRITICAL: Uses expanding window for first 'window' periods to avoid NaN,
        then switches to rolling window. This ensures no forward-looking data is used.

        Parameters:
        -----------
        series : pd.Series
            Input feature series
        window : int
            Rolling window size

        Returns:
        --------
        pd.Series : Z-score normalized series
        """
        roll_mean = series.rolling(window=window, min_periods=window).mean()
        roll_std = series.rolling(window=window, min_periods=window).std()
        roll_std = roll_std.replace(0, 1e-06)
        zscore = (series - roll_mean) / roll_std
        return zscore

    def calculate_features(self, df: pd.DataFrame, config: Config) -> pd.DataFrame:
        """
        Calculate all features for GMM model.

        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        config : Config
            Configuration object with parameters

        Returns:
        --------
        pd.DataFrame : Original data with added feature columns
        """
        df = df.copy()
        print("  Calculating Stochastic K...")
        df["stoch_k"] = self.calculate_stochastic_k(
            df, period=config.STOCHASTIC_PERIOD, smooth=config.STOCHASTIC_SMOOTH
        )
        print("  Calculating Garman-Klass Volatility...")
        df["gk_volatility"] = self.calculate_garman_klass_volatility(
            df, period=config.GARMAN_KLASS_PERIOD
        )
        print("  Normalizing features with rolling z-score...")
        df["stoch_k_norm"] = self.rolling_zscore(df["stoch_k"], window=config.ZSCORE_LOOKBACK)
        df["gk_vol_norm"] = self.rolling_zscore(df["gk_volatility"], window=config.ZSCORE_LOOKBACK)
        return df


class RegimeDetector:
    """
    Detects market regimes using Gaussian Mixture Model on normalized features.
    Uses expanding window training to avoid look-ahead bias.

    ENHANCEMENT: Classifies regimes as bullish, neutral, or bearish based on
    historical performance characteristics.
    """

    def __init__(self, config: Config):
        self.config = config
        self.models = {}
        self.last_train_date = {}

    def train_gmm(self, features: np.ndarray) -> GaussianMixture:
        """
        Train a Gaussian Mixture Model on feature data.

        Parameters:
        -----------
        features : np.ndarray
            Feature matrix (n_samples, n_features)

        Returns:
        --------
        GaussianMixture : Trained GMM model
        """
        gmm = GaussianMixture(
            n_components=self.config.N_COMPONENTS,
            covariance_type=self.config.GMM_COVARIANCE,
            max_iter=self.config.GMM_MAX_ITER,
            n_init=self.config.GMM_N_INIT,
            random_state=self.config.GMM_RANDOM_STATE,
        )
        gmm.fit(features)
        return gmm

    def predict_regime(self, df: pd.DataFrame, ticker: str) -> pd.DataFrame:
        """
        Predict regime for each day using expanding window training.

        CRITICAL: At time t, we only use data up to t-1 for training,
        then predict regime for time t. This ensures no look-ahead bias.

        Signal Flow:
        1. At Close[t], detect regime using features up to Close[t]
        2. Regime determines which instrument to trade at Open[t+1]
        3. Position held from Open[t+1] until next regime change

        Parameters:
        -----------
        df : pd.DataFrame
            Feature data with 'stoch_k_norm' and 'gk_vol_norm' columns
        ticker : str
            Ticker symbol (for model storage)

        Returns:
        --------
        pd.DataFrame : Data with added 'regime' and 'regime_proba' columns
        """
        df = df.copy()
        df["regime"] = np.nan
        df["regime_proba"] = np.nan
        feature_cols = ["stoch_k_norm", "gk_vol_norm"]
        valid_idx = df[feature_cols].dropna().index
        if len(valid_idx) < self.config.MIN_TRAINING_SAMPLES:
            print(f"Insufficient data for {ticker}: {len(valid_idx)} samples")
            return df
        train_counter = 0
        current_model = None
        for i, date in enumerate(valid_idx):
            train_data = df.loc[valid_idx[:i], feature_cols].values
            if len(train_data) < self.config.MIN_TRAINING_SAMPLES:
                continue
            if current_model is None or train_counter >= self.config.RETRAIN_FREQUENCY:
                print(
                    f"  Training GMM at {date.strftime('%Y-%m-%d')} with {len(train_data)} samples"
                )
                current_model = self.train_gmm(train_data)
                train_counter = 0
            current_features = df.loc[date, feature_cols].values.reshape(1, -1)
            regime = current_model.predict(current_features)[0]
            proba = current_model.predict_proba(current_features)[0]
            regime_probability = proba[regime]
            df.loc[date, "regime"] = regime
            df.loc[date, "regime_proba"] = regime_probability
            train_counter += 1
        return df

    def analyze_regimes(self, df: pd.DataFrame) -> Dict:
        """
        Analyze characteristics of each detected regime to classify as
        bullish, neutral, or bearish.

        CRITICAL: This analysis is done post-hoc to understand regime characteristics.
        It uses forward returns for analysis ONLY, not for signal generation.

        Parameters:
        -----------
        df : pd.DataFrame
            Data with regime labels

        Returns:
        --------
        Dict : Regime statistics including classification
        """
        regime_stats = {}
        for regime in range(self.config.N_COMPONENTS):
            regime_data = df[df["regime"] == regime].copy()
            if len(regime_data) == 0:
                continue
            regime_data["forward_return"] = regime_data["Close"].pct_change().shift(-1)
            stats = {
                "count": len(regime_data),
                "pct_of_total": len(regime_data) / len(df) * 100,
                "mean_return": regime_data["forward_return"].mean() * 100,
                "volatility": regime_data["forward_return"].std() * np.sqrt(252) * 100,
                "sharpe": (
                    regime_data["forward_return"].mean()
                    / regime_data["forward_return"].std()
                    * np.sqrt(252)
                    if regime_data["forward_return"].std() > 0
                    else 0
                ),
                "mean_stoch_k": regime_data["stoch_k"].mean(),
                "mean_gk_vol": regime_data["gk_volatility"].mean(),
                "win_rate": (regime_data["forward_return"] > 0).sum() / len(regime_data) * 100,
            }
            regime_stats[regime] = stats
        return regime_stats

    def classify_regimes(self, regime_stats: Dict, config: Config) -> Dict[int, str]:
        """
        Classify each GMM regime as 'bullish', 'neutral', or 'bearish'
        based on historical Sharpe ratio.

        Classification Logic:
        - Sharpe > BULLISH_SHARPE_THRESHOLD (default 0.5): BULLISH
        - Sharpe < BEARISH_SHARPE_THRESHOLD (default -0.2): BEARISH
        - Otherwise: NEUTRAL

        CRITICAL: This classification is based on historical performance during
        the training period. It answers: "When we were in this regime historically,
        what happened to returns?"

        Parameters:
        -----------
        regime_stats : Dict
            Statistics for each regime
        config : Config
            Configuration with thresholds

        Returns:
        --------
        Dict[int, str] : Mapping from regime number to classification
        """
        regime_classification = {}
        for regime, stats in regime_stats.items():
            sharpe = stats["sharpe"]
            if sharpe > config.BULLISH_SHARPE_THRESHOLD:
                regime_classification[regime] = "bullish"
            elif sharpe < config.BEARISH_SHARPE_THRESHOLD:
                regime_classification[regime] = "bearish"
            else:
                regime_classification[regime] = "neutral"
        return regime_classification

    def assign_instruments_and_positions(
        self, regime_classification: Dict[int, str], config: Config
    ) -> Tuple[Dict[int, str], Dict[int, float]]:
        """
        Assign trading instrument and position size for each regime based on classification.

        Mapping:
        - BULLISH regime → Trade TQQQ at BULLISH_POSITION_SIZE (default 100%)
        - NEUTRAL regime → Trade SPY at NEUTRAL_POSITION_SIZE (default 100%)
        - BEARISH regime → Trade SH at BEARISH_POSITION_SIZE (default 20% for hedging)

        CRITICAL: This creates the trading rules that will be applied at Open[t+1]
        after regime detection at Close[t].

        Parameters:
        -----------
        regime_classification : Dict[int, str]
            Classification of each regime
        config : Config
            Configuration with instruments and position sizes

        Returns:
        --------
        Tuple[Dict[int, str], Dict[int, float]]:
            - Mapping from regime to instrument ticker
            - Mapping from regime to position size
        """
        regime_instruments = {}
        regime_positions = {}
        for regime, classification in regime_classification.items():
            if classification == "bullish":
                regime_instruments[regime] = config.BULLISH_INSTRUMENT
                regime_positions[regime] = config.BULLISH_POSITION_SIZE
            elif classification == "neutral":
                regime_instruments[regime] = config.NEUTRAL_INSTRUMENT
                regime_positions[regime] = config.NEUTRAL_POSITION_SIZE
            else:
                regime_instruments[regime] = config.BEARISH_INSTRUMENT
                regime_positions[regime] = config.BEARISH_POSITION_SIZE
        return (regime_instruments, regime_positions)


class Backtester:
    """
    Backtesting engine for multi-instrument regime strategy.

    Trading Flow:
    1. At Close[t]: Detect regime from features
    2. At Close[t]: Determine which instrument to trade based on regime
    3. At Open[t+1]: Execute trade in selected instrument
    4. Hold position until regime changes
    5. Returns calculated from Close[t] to Close[t+1]

    CRITICAL ENHANCEMENTS:
    - Handles switching between different instruments (TQQQ, SPY, SH)
    - Accounts for transaction costs when switching instruments
    - Properly aligns regime detection with instrument trading
    """

    def __init__(self, config: Config):
        self.config = config

    def merge_instrument_data(
        self, regime_df: pd.DataFrame, instruments_data: Dict[str, pd.DataFrame]
    ) -> pd.DataFrame:
        """
        Merge regime detections with price data from multiple instruments.

        Creates a unified DataFrame with:
        - Regime detections from FEATURE_TICKER
        - Price data from all trading instruments (TQQQ, SPY, SH)

        CRITICAL: Aligns dates properly to avoid look-ahead bias.
        Regime detected at Close[t] → Trade instrument at Open[t+1]

        Parameters:
        -----------
        regime_df : pd.DataFrame
            Features and regime detections from FEATURE_TICKER
        instruments_data : Dict[str, pd.DataFrame]
            OHLCV data for each trading instrument

        Returns:
        --------
        pd.DataFrame : Merged data with regime and all instrument prices
        """
        merged = regime_df[["regime", "regime_proba"]].copy()
        for ticker, df in instruments_data.items():
            ticker_suffix = ticker.replace("-", "_")
            for col in ["Open", "High", "Low", "Close", "Volume"]:
                merged[f"{ticker_suffix}_{col}"] = df[col]
        return merged

    def generate_signals(
        self,
        df: pd.DataFrame,
        regime_instruments: Dict[int, str],
        regime_positions: Dict[int, float],
        config: Config,
    ) -> pd.DataFrame:
        """
        Generate trading signals based on regime detection.

        Signal Logic:
        1. Regime detected at Close[t]
        2. Map regime → instrument and position size
        3. Signal indicates: which instrument to hold and at what % of portfolio
        4. Execution occurs at Open[t+1]

        CRITICAL: Applies confidence filter to avoid trading on uncertain regimes.
        Low confidence → reduce position to 0% (go to cash)

        Parameters:
        -----------
        df : pd.DataFrame
            Data with regime column and regime_proba column
        regime_instruments : Dict[int, str]
            Mapping from regime to instrument ticker
        regime_positions : Dict[int, float]
            Mapping from regime to position size
        config : Config
            Configuration object

        Returns:
        --------
        pd.DataFrame : Data with 'signal_instrument' and 'signal_position' columns
        """
        df = df.copy()
        df["signal_instrument"] = df["regime"].map(regime_instruments)
        df["signal_position"] = df["regime"].map(regime_positions)
        if config.USE_REGIME_CONFIDENCE and "regime_proba" in df.columns:
            low_confidence_mask = df["regime_proba"] < config.MIN_REGIME_PROBABILITY
            n_filtered = low_confidence_mask.sum()
            print(
                f"  Confidence filter: {n_filtered} / {len(df)} days below {config.MIN_REGIME_PROBABILITY} threshold"
            )
            df.loc[low_confidence_mask, "signal_position"] = 0.0
            df.loc[low_confidence_mask, "signal_instrument"] = None
        df["signal_position"] = df["signal_position"].fillna(0)
        return df

    def calculate_returns(self, df: pd.DataFrame, config: Config) -> pd.DataFrame:
        """
        Calculate strategy returns accounting for:
        1. Regime-based instrument selection
        2. Position sizing per regime
        3. Transaction costs (including instrument switches)
        4. Proper return calculation: Close[t] to Close[t+1]

        CRITICAL LOGIC:
        - Signal at Close[t] determines instrument and position for period [t+1]
        - Position held from Open[t+1] to Open[t+2] (or Close[t+1] for simplicity)
        - Returns = Close[t+1] / Close[t] - 1, weighted by position
        - Transaction costs applied when position OR instrument changes

        Parameters:
        -----------
        df : pd.DataFrame
            Data with signal columns and instrument price columns
        config : Config
            Configuration object

        Returns:
        --------
        pd.DataFrame : Data with returns calculations
        """
        df = df.copy()
        for instrument in [
            config.BULLISH_INSTRUMENT,
            config.NEUTRAL_INSTRUMENT,
            config.BEARISH_INSTRUMENT,
        ]:
            ticker_suffix = instrument.replace("-", "_")
            close_col = f"{ticker_suffix}_Close"
            if close_col in df.columns:
                df[f"{ticker_suffix}_returns"] = df[close_col].pct_change()
        df["position"] = df["signal_position"].shift(1)
        df["instrument_held"] = df["signal_instrument"].shift(1)
        df["strategy_returns"] = 0.0
        for instrument in [
            config.BULLISH_INSTRUMENT,
            config.NEUTRAL_INSTRUMENT,
            config.BEARISH_INSTRUMENT,
        ]:
            ticker_suffix = instrument.replace("-", "_")
            returns_col = f"{ticker_suffix}_returns"
            if returns_col in df.columns:
                mask = df["instrument_held"] == instrument
                df.loc[mask, "strategy_returns"] = (
                    df.loc[mask, "position"] * df.loc[mask, returns_col]
                )
        df["position_change"] = df["position"].diff().abs()
        df["instrument_change"] = (df["instrument_held"] != df["instrument_held"].shift(1)).astype(
            float
        )
        df["transaction_costs"] = 0.0
        df["transaction_costs"] += df["position_change"] * config.COMMISSION_RATE
        instrument_switch_mask = (df["instrument_change"] > 0) & (df["position"] > 0)
        df.loc[instrument_switch_mask, "transaction_costs"] += (
            df.loc[instrument_switch_mask, "position"] * config.COMMISSION_RATE
        )
        df["strategy_returns_net"] = df["strategy_returns"] - df["transaction_costs"]
        df["cum_returns"] = (1 + df["strategy_returns_net"].fillna(0)).cumprod()
        return df

    def backtest_multi_instrument_strategy(
        self,
        regime_df: pd.DataFrame,
        instruments_data: Dict[str, pd.DataFrame],
        regime_instruments: Dict[int, str],
        regime_positions: Dict[int, float],
        config: Config,
    ) -> pd.DataFrame:
        """
        Run complete backtest for multi-instrument regime strategy.

        Process:
        1. Merge regime data with instrument price data
        2. Generate trading signals based on regime
        3. Calculate returns with proper instrument selection
        4. Account for all transaction costs

        Parameters:
        -----------
        regime_df : pd.DataFrame
            Regime detections from feature ticker
        instruments_data : Dict[str, pd.DataFrame]
            Price data for all trading instruments
        regime_instruments : Dict[int, str]
            Mapping from regime to instrument
        regime_positions : Dict[int, float]
            Mapping from regime to position size
        config : Config
            Configuration object

        Returns:
        --------
        pd.DataFrame : Complete backtest results
        """
        print("\n  Merging regime data with instrument prices...")
        df = self.merge_instrument_data(regime_df, instruments_data)
        print("  Generating trading signals...")
        df = self.generate_signals(df, regime_instruments, regime_positions, config)
        print("  Calculating returns and costs...")
        df = self.calculate_returns(df, config)
        return df


class PerformanceAnalyzer:
    """
    Calculate and display performance metrics for strategy evaluation.
    """

    @staticmethod
    def calculate_metrics(returns: pd.Series, name: str = "Strategy") -> Dict:
        """
        Calculate comprehensive performance metrics.

        Parameters:
        -----------
        returns : pd.Series
            Series of returns
        name : str
            Name of the strategy/asset

        Returns:
        --------
        Dict : Performance metrics
        """
        returns = returns.dropna()
        if len(returns) == 0:
            return {}
        total_return = (1 + returns).prod() - 1
        ann_return = (1 + total_return) ** (252 / len(returns)) - 1
        ann_volatility = returns.std() * np.sqrt(252)
        sharpe_ratio = ann_return / ann_volatility if ann_volatility > 0 else 0
        cum_returns = (1 + returns).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        max_drawdown = drawdown.min()
        win_rate = (returns > 0).sum() / len(returns) * 100
        metrics = {
            "name": name,
            "total_return_%": total_return * 100,
            "annual_return_%": ann_return * 100,
            "annual_volatility_%": ann_volatility * 100,
            "sharpe_ratio": sharpe_ratio,
            "max_drawdown_%": max_drawdown * 100,
            "win_rate_%": win_rate,
            "n_trades": len(returns),
        }
        return metrics

    @staticmethod
    def print_metrics(metrics: Dict):
        """Pretty print performance metrics."""
        print(f"\n{'=' * 60}")
        print(f"Performance Metrics: {metrics['name']}")
        print(f"{'=' * 60}")
        print(f"Total Return:          {metrics['total_return_%']:>10.2f}%")
        print(f"Annual Return:         {metrics['annual_return_%']:>10.2f}%")
        print(f"Annual Volatility:     {metrics['annual_volatility_%']:>10.2f}%")
        print(f"Sharpe Ratio:          {metrics['sharpe_ratio']:>10.2f}")
        print(f"Max Drawdown:          {metrics['max_drawdown_%']:>10.2f}%")
        print(f"Win Rate:              {metrics['win_rate_%']:>10.2f}%")
        print(f"Number of Periods:     {metrics['n_trades']:>10}")
        print(f"{'=' * 60}\n")

    @staticmethod
    def plot_results(strategy_df: pd.DataFrame, benchmark_df: pd.DataFrame, config: Config):
        """
        Plot backtest results comparing strategy vs benchmark.

        Enhanced to show regime-based instrument allocation.
        """
        fig, axes = plt.subplots(4, 1, figsize=(14, 12))
        axes[0].plot(
            strategy_df.index,
            strategy_df["cum_returns"],
            label="Multi-Instrument GMM Strategy",
            linewidth=2,
            color="darkblue",
        )
        axes[0].plot(
            benchmark_df.index,
            benchmark_df["cum_returns"],
            label=f"{config.BENCHMARK_TICKER} Benchmark",
            linewidth=2,
            alpha=0.7,
            color="gray",
        )
        axes[0].set_ylabel("Cumulative Returns")
        axes[0].set_title("Strategy Performance vs Benchmark", fontweight="bold")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        axes[1].set_ylabel("Instrument Held")
        axes[1].set_title("Instrument Selection by Regime", fontweight="bold")
        for i, instrument in enumerate(
            [config.BULLISH_INSTRUMENT, config.NEUTRAL_INSTRUMENT, config.BEARISH_INSTRUMENT]
        ):
            mask = strategy_df["instrument_held"] == instrument
            if mask.any():
                axes[1].scatter(
                    strategy_df[mask].index, [i] * mask.sum(), alpha=0.6, s=10, label=instrument
                )
        axes[1].set_yticks([0, 1, 2])
        axes[1].set_yticklabels(
            [config.BULLISH_INSTRUMENT, config.NEUTRAL_INSTRUMENT, config.BEARISH_INSTRUMENT]
        )
        axes[1].legend(loc="upper right")
        axes[1].grid(True, alpha=0.3, axis="x")
        axes[2].fill_between(
            strategy_df.index,
            strategy_df["position"] * 100,
            0,
            alpha=0.3,
            color="green",
            label="Position Size",
        )
        axes[2].plot(
            strategy_df.index, strategy_df["position"] * 100, linewidth=1, color="darkgreen"
        )
        axes[2].set_ylabel("Position Size (%)")
        axes[2].set_title("Portfolio Exposure Over Time", fontweight="bold")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
        strategy_rolling_sharpe = (
            strategy_df["strategy_returns_net"].rolling(252).mean()
            / strategy_df["strategy_returns_net"].rolling(252).std()
            * np.sqrt(252)
        )
        benchmark_rolling_sharpe = (
            benchmark_df["returns"].rolling(252).mean()
            / benchmark_df["returns"].rolling(252).std()
            * np.sqrt(252)
        )
        axes[3].plot(
            strategy_df.index,
            strategy_rolling_sharpe,
            label="Multi-Instrument Strategy",
            linewidth=2,
            color="darkblue",
        )
        axes[3].plot(
            benchmark_df.index,
            benchmark_rolling_sharpe,
            label=f"{config.BENCHMARK_TICKER} Benchmark",
            linewidth=2,
            alpha=0.7,
            color="gray",
        )
        axes[3].axhline(y=0, color="black", linestyle="--", alpha=0.3)
        axes[3].set_ylabel("Rolling Sharpe (252d)")
        axes[3].set_xlabel("Date")
        axes[3].set_title("Rolling Sharpe Ratio", fontweight="bold")
        axes[3].legend()
        axes[3].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_regime_analysis(
        df: pd.DataFrame, config: Config, regime_classification: Dict[int, str]
    ):
        """
        Plot detailed regime analysis showing classification and characteristics.

        Enhanced to show which instrument is used for each regime.
        """
        df_clean = df.dropna(subset=["regime"])
        if len(df_clean) == 0:
            print("No valid regime data")
            return
        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
        regime_colors = {"bullish": "green", "neutral": "blue", "bearish": "red"}
        ax1 = fig.add_subplot(gs[0, :])
        regime_counts = df_clean["regime"].value_counts().sort_index()
        colors = [
            regime_colors.get(regime_classification.get(int(r), "neutral"), "gray")
            for r in regime_counts.index
        ]
        labels = [
            f"Regime {int(r)}\n({regime_classification.get(int(r), 'unknown')})"
            for r in regime_counts.index
        ]
        bars = ax1.bar(range(len(regime_counts)), regime_counts.values, color=colors, alpha=0.7)
        ax1.set_xticks(range(len(regime_counts)))
        ax1.set_xticklabels(labels)
        ax1.set_ylabel("Number of Days")
        ax1.set_title("Regime Distribution and Classification", fontweight="bold", fontsize=14)
        ax1.grid(True, alpha=0.3, axis="y")
        for bar in bars:
            height = bar.get_height()
            ax1.text(
                bar.get_x() + bar.get_width() / 2.0,
                height,
                f"{int(height)}",
                ha="center",
                va="bottom",
            )
        ax2 = fig.add_subplot(gs[1, 0])
        for regime in df_clean["regime"].unique():
            if pd.notna(regime):
                regime_data = df_clean[df_clean["regime"] == regime]
                classification = regime_classification.get(int(regime), "neutral")
                color = regime_colors.get(classification, "gray")
                ax2.scatter(
                    regime_data.index,
                    [regime] * len(regime_data),
                    alpha=0.5,
                    s=15,
                    color=color,
                    label=f"Regime {int(regime)} ({classification})",
                )
        ax2.set_ylabel("Regime")
        ax2.set_title("Regime Transitions Over Time", fontweight="bold")
        ax2.legend(loc="upper right", fontsize=8)
        ax2.grid(True, alpha=0.3)
        ax3 = fig.add_subplot(gs[1, 1])
        instrument_counts = df_clean["instrument_held"].value_counts()
        if len(instrument_counts) > 0:
            colors_inst = []
            for inst in instrument_counts.index:
                if inst == config.BULLISH_INSTRUMENT:
                    colors_inst.append("green")
                elif inst == config.BEARISH_INSTRUMENT:
                    colors_inst.append("red")
                else:
                    colors_inst.append("blue")
            bars = ax3.bar(
                range(len(instrument_counts)),
                instrument_counts.values,
                color=colors_inst,
                alpha=0.7,
            )
            ax3.set_xticks(range(len(instrument_counts)))
            ax3.set_xticklabels(instrument_counts.index, rotation=45)
            ax3.set_ylabel("Number of Days")
            ax3.set_title("Instrument Usage Frequency", fontweight="bold")
            ax3.grid(True, alpha=0.3, axis="y")
            for bar in bars:
                height = bar.get_height()
                ax3.text(
                    bar.get_x() + bar.get_width() / 2.0,
                    height,
                    f"{int(height)}",
                    ha="center",
                    va="bottom",
                )
        ax4 = fig.add_subplot(gs[2, :])
        for instrument in [
            config.BULLISH_INSTRUMENT,
            config.NEUTRAL_INSTRUMENT,
            config.BEARISH_INSTRUMENT,
        ]:
            mask = df_clean["instrument_held"] == instrument
            if mask.any():
                instrument_data = df_clean[mask].copy()
                instrument_data["inst_cum_ret"] = (
                    1 + instrument_data["strategy_returns_net"].fillna(0)
                ).cumprod()
                color = (
                    "green"
                    if instrument == config.BULLISH_INSTRUMENT
                    else "red" if instrument == config.BEARISH_INSTRUMENT else "blue"
                )
                ax4.plot(
                    instrument_data.index,
                    instrument_data["inst_cum_ret"],
                    label=instrument,
                    linewidth=2,
                    alpha=0.7,
                    color=color,
                )
        ax4.set_ylabel("Cumulative Returns")
        ax4.set_xlabel("Date")
        ax4.set_title("Cumulative Returns by Instrument Held", fontweight="bold")
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        plt.suptitle(f"Multi-Instrument Regime Strategy Analysis", fontsize=16, fontweight="bold")
        plt.show()

    @staticmethod
    def print_strategy_analysis(
        df: pd.DataFrame,
        config: Config,
        regime_classification: Dict[int, str],
        regime_instruments: Dict[int, str],
    ):
        """
        Print detailed strategy analysis including regime and instrument statistics.
        """
        print("\n" + "=" * 80)
        print("DETAILED STRATEGY ANALYSIS")
        print("=" * 80)
        n_trades = (df["position"].diff() != 0).sum()
        instrument_prev = df["instrument_held"].shift(1)
        n_instrument_switches = (df["instrument_held"] != instrument_prev).sum()
        avg_position = df["position"].mean() * 100
        print(f"\nTrading Activity:")
        print(f"  Total Position Changes:     {n_trades}")
        print(f"  Instrument Switches:        {n_instrument_switches}")
        print(f"  Average Position Size:      {avg_position:.1f}%")
        print(f"\nRegime Distribution:")
        regime_dist = df["regime"].value_counts().sort_index()
        for regime in regime_dist.index:
            if pd.notna(regime):
                regime_int = int(regime)
                count = regime_dist[regime]
                pct = count / len(df) * 100
                classification = regime_classification.get(regime_int, "unknown")
                instrument = regime_instruments.get(regime_int, "unknown")
                regime_data = df[df["regime"] == regime]
                avg_pos = regime_data["position"].mean() * 100
                print(f"\n  Regime {regime_int} ({classification.upper()}):")
                print(f"    Days:              {count} ({pct:.1f}%)")
                print(f"    Instrument:        {instrument}")
                print(f"    Avg Position:      {avg_pos:.1f}%")
        print(f"\nInstrument Usage:")
        instrument_dist = df["instrument_held"].value_counts()
        for instrument, count in instrument_dist.items():
            if pd.notna(instrument):
                pct = count / len(df) * 100
                instrument_data = df[df["instrument_held"] == instrument]
                avg_ret = instrument_data["strategy_returns_net"].mean() * 252 * 100
                vol = instrument_data["strategy_returns_net"].std() * np.sqrt(252) * 100
                sharpe = avg_ret / vol if vol > 0 else 0
                print(f"\n  {instrument}:")
                print(f"    Days Held:         {count} ({pct:.1f}%)")
                print(f"    Avg Ann. Return:   {avg_ret:.2f}%")
                print(f"    Ann. Volatility:   {vol:.2f}%")
                print(f"    Sharpe Ratio:      {sharpe:.2f}")
        if "regime_proba" in df.columns:
            avg_confidence = df["regime_proba"].mean() * 100
            low_conf_days = (df["regime_proba"] < config.MIN_REGIME_PROBABILITY).sum()
            print(f"\nRegime Confidence:")
            print(f"  Average Confidence:        {avg_confidence:.1f}%")
            print(
                f"  Days Below Threshold:      {low_conf_days} ({low_conf_days / len(df) * 100:.1f}%)"
            )
        print("\n" + "=" * 80)


def main():
    """
    Main execution function for multi-instrument regime strategy.
    """
    print("=" * 80)
    print("GMM REGIME DETECTION - MULTI-INSTRUMENT ADAPTIVE STRATEGY")
    print("=" * 80)
    print("\nStrategy Overview:")
    print("  - Detects market regimes using GMM on SPY features")
    print("  - BULLISH regimes → Trade TQQQ (3x leveraged)")
    print("  - NEUTRAL regimes → Trade SPY (standard exposure)")
    print("  - BEARISH regimes → Trade SH (inverse ETF for hedging)")
    print("=" * 80)
    config = Config()
    print("\n[1/7] Fetching market data from Yahoo Finance...")
    fetcher = FMPDataFetcher()
    print(f"\nFetching feature data ({config.FEATURE_TICKER})...")
    feature_data = fetcher.get_historical_data(
        config.FEATURE_TICKER, config.START_DATE, config.END_DATE
    )
    if feature_data is None:
        print(f"ERROR: Could not fetch data for {config.FEATURE_TICKER}")
        return
    print("\nFetching trading instruments...")
    instruments_data = {}
    for ticker in [config.BULLISH_INSTRUMENT, config.NEUTRAL_INSTRUMENT, config.BEARISH_INSTRUMENT]:
        print(f"  Fetching {ticker}...")
        df = fetcher.get_historical_data(ticker, config.START_DATE, config.END_DATE)
        if df is not None:
            instruments_data[ticker] = df
        else:
            print(f"  WARNING: Could not fetch {ticker}")
    print(f"\nFetching benchmark data ({config.BENCHMARK_TICKER})...")
    benchmark_data = fetcher.get_historical_data(
        config.BENCHMARK_TICKER, config.START_DATE, config.END_DATE
    )
    if benchmark_data is None:
        print(f"ERROR: Could not fetch benchmark data for {config.BENCHMARK_TICKER}")
        return
    print("\n[2/7] Engineering features from feature ticker...")
    engineer = FeatureEngineer()
    print(f"\nCalculating features for {config.FEATURE_TICKER}...")
    feature_data = engineer.calculate_features(feature_data, config)
    print(f"\nCalculating features for benchmark...")
    benchmark_data = engineer.calculate_features(benchmark_data, config)
    print("\n[3/7] Detecting market regimes using GMM...")
    detector = RegimeDetector(config)
    print(f"\nDetecting regimes from {config.FEATURE_TICKER} features...")
    feature_data = detector.predict_regime(feature_data, config.FEATURE_TICKER)
    print("\n[4/7] Analyzing and classifying regimes...")
    regime_stats = detector.analyze_regimes(feature_data)
    print("\nRegime Statistics:")
    print("=" * 80)
    for regime, stats in regime_stats.items():
        print(f"\nRegime {regime}:")
        print(f"  Days:                  {stats['count']:.0f} ({stats['pct_of_total']:.1f}%)")
        print(f"  Mean Forward Return:   {stats['mean_return']:.3f}%")
        print(f"  Volatility:            {stats['volatility']:.2f}%")
        print(f"  Sharpe Ratio:          {stats['sharpe']:.2f}")
        print(f"  Win Rate:              {stats['win_rate']:.2f}%")
        print(f"  Mean Stochastic K:     {stats['mean_stoch_k']:.2f}")
        print(f"  Mean GK Volatility:    {stats['mean_gk_vol']:.2f}")
    regime_classification = detector.classify_regimes(regime_stats, config)
    print("\n" + "=" * 80)
    print("REGIME CLASSIFICATION")
    print("=" * 80)
    for regime, classification in regime_classification.items():
        sharpe = regime_stats[regime]["sharpe"]
        print(f"  Regime {regime}: {classification.upper()} (Sharpe: {sharpe:.2f})")
    regime_instruments, regime_positions = detector.assign_instruments_and_positions(
        regime_classification, config
    )
    print("\n" + "=" * 80)
    print("TRADING RULES")
    print("=" * 80)
    for regime in sorted(regime_instruments.keys()):
        classification = regime_classification[regime]
        instrument = regime_instruments[regime]
        position = regime_positions[regime]
        print(f"\n  Regime {regime} ({classification.upper()}):")
        print(f"    Instrument:  {instrument}")
        print(f"    Position:    {position * 100:.1f}%")
    print("\n[5/7] Running multi-instrument backtest...")
    backtester = Backtester(config)
    backtest_results = backtester.backtest_multi_instrument_strategy(
        feature_data, instruments_data, regime_instruments, regime_positions, config
    )
    print(f"\nBacktesting benchmark ({config.BENCHMARK_TICKER})...")
    benchmark_data["returns"] = benchmark_data["Close"].pct_change()
    benchmark_data["cum_returns"] = (1 + benchmark_data["returns"]).cumprod()
    print("\n[6/7] Analyzing performance...")
    analyzer = PerformanceAnalyzer()
    strategy_metrics = analyzer.calculate_metrics(
        backtest_results["strategy_returns_net"], "Multi-Instrument GMM Strategy"
    )
    benchmark_metrics = analyzer.calculate_metrics(
        benchmark_data["returns"], f"{config.BENCHMARK_TICKER} Benchmark"
    )
    analyzer.print_metrics(strategy_metrics)
    analyzer.print_metrics(benchmark_metrics)
    analyzer.print_strategy_analysis(
        backtest_results, config, regime_classification, regime_instruments
    )
    if config.PLOT_RESULTS:
        print("\n[7/7] Generating visualizations...")
        analyzer.plot_results(backtest_results, benchmark_data, config)
        analyzer.plot_regime_analysis(backtest_results, config, regime_classification)
    print("\n" + "=" * 80)
    print("BACKTEST COMPLETE")
    print("=" * 80)
    return {
        "backtest_results": backtest_results,
        "benchmark_data": benchmark_data,
        "strategy_metrics": strategy_metrics,
        "benchmark_metrics": benchmark_metrics,
        "regime_classification": regime_classification,
        "regime_instruments": regime_instruments,
        "regime_positions": regime_positions,
    }


if __name__ == "__main__":
    results = main()
